# Evaluating Chain-of-Thought Prompting and Self-Consistency for Multi-Step Reasoning in Small Open-Weight Language Models

In [ ]:
!pip install -q --upgrade transformers accelerate datasets vllm

In [ ]:
import os, re, json, random, time
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams
import matplotlib.pyplot as plt

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = Path('/content/drive/MyDrive/coms4705_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
MODEL_NAME       = "Qwen/Qwen2.5-1.5B-Instruct"

N_EVAL = 200
SC_SAMPLES = 20
SC_TEMPERATURE = 0.7
COT_TEMPERATURE = 0.0
MAX_NEW_TOKENS = 320
SEED = 42
RESULTS_DIR = Path('/content/drive/MyDrive/coms4705_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)

## Model loading


In [ ]:
llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
)
tokenizer = llm.get_tokenizer()
print(f"Loaded {MODEL_NAME} with vLLM")

## Dataset and evaluation

In [ ]:
gsm8k = load_dataset("gsm8k", "main")["test"]
print(f"GSM8K test size: {len(gsm8k)}")

indices = random.Random(SEED).sample(range(len(gsm8k)), N_EVAL)
eval_set = [gsm8k[i] for i in indices]
print(f"Evaluating on {len(eval_set)} problems (seed={SEED})")
print("\nExample problem:")
print(eval_set[0]["question"])
print("\nGold solution:")
print(eval_set[0]["answer"])


###Prompt templates

In [ ]:
# 8 CoT exemplars from Wei et al 2022.
COT_EXEMPLARS = [
    {
        "q": "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?",
        "a": "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.",
    },
    {
        "q": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?",
        "a": "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.",
    },
    {
        "q": "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        "a": "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The answer is 39.",
    },
    {
        "q": "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?",
        "a": "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The answer is 8.",
    },
    {
        "q": "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?",
        "a": "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The answer is 9.",
    },
    {
        "q": "There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?",
        "a": "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 = 29. The answer is 29.",
    },
    {
        "q": "Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?",
        "a": "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The answer is 33.",
    },
    {
        "q": "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?",
        "a": "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 = 8 dollars left. The answer is 8.",
    },
]

def _short_answer(rationale_with_answer: str) -> str:
    m = re.search(r"The answer is\s*([^.]+)\.", rationale_with_answer)
    return f"The answer is {m.group(1).strip()}." if m else rationale_with_answer

IO_EXEMPLARS = [{"q": ex["q"], "a": _short_answer(ex["a"])} for ex in COT_EXEMPLARS]

def build_prompt(question: str, exemplars: list) -> str:
    shots = "\n\n".join(f"Q: {ex['q']}\nA: {ex['a']}" for ex in exemplars)
    return f"{shots}\n\nQ: {question}\nA:"

def build_standard_prompt(q):  return build_prompt(q, IO_EXEMPLARS)
def build_cot_prompt(q):       return build_prompt(q, COT_EXEMPLARS)
def build_zeroshot_cot_prompt(q): return f"Q: {q}\nA: Let's think step by step."

# Sanity check
print(build_cot_prompt("Sample question?")[:600], "...")

## Answer extraction


In [ ]:
_NUM = r"-?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?"

def extract_gold(answer_field: str):
    m = re.search(r"####\s*(" + _NUM + ")", answer_field)
    return _to_float(m.group(1)) if m else None

def extract_pred(generation: str):
    text = generation.split("\nQ:")[0]
    m = re.search(r"answer\s*is\s*\$?\s*(" + _NUM + ")", text, re.IGNORECASE)
    if m:
        return _to_float(m.group(1))
    nums = re.findall(_NUM, text)
    return _to_float(nums[-1]) if nums else None

def _to_float(s):
    if s is None: return None
    try: return float(str(s).replace(",", "").replace("$", "").strip())
    except ValueError: return None

def is_correct(pred, gold, tol=1e-4):
    return pred is not None and gold is not None and abs(pred - gold) < tol

assert extract_gold("Janet ... #### 18") == 18.0
assert extract_gold("Total cost #### 1200") == 1200.0
assert extract_gold("Total cost #### 1,200") == 1200.0
assert extract_pred("Step 1 ... So the answer is 42.") == 42.0
assert extract_pred("She has $1,200 left. The answer is $1,200.") == 1200.0
print("Extraction unit tests passed.")

## Evaluation loop

In [ ]:
def eval_condition(eval_set, build_prompt_fn, *, n_samples=1, temperature=0.0,
                   max_tokens=320, label=""):
    prompts = [build_prompt_fn(ex["question"]) for ex in eval_set]

    params = SamplingParams(
        temperature=temperature,
        top_p=0.95 if temperature > 0 else 1.0,
        max_tokens=max_tokens,
        n=n_samples,
        stop=["\nQ:"],
        seed=SEED if temperature == 0 else None,
    )

    t0 = time.time()
    outputs = llm.generate(prompts, params)
    elapsed = time.time() - t0

    correct = 0
    records = []
    for ex, output in zip(eval_set, outputs):
        gens = [o.text.strip() for o in output.outputs]
        preds = [extract_pred(g) for g in gens]
        gold = extract_gold(ex["answer"])

        if n_samples == 1:
            final_pred = preds[0]
            chosen_idx = 0
        else:
            valid = [p for p in preds if p is not None]
            if valid:
                most_common, _ = Counter(valid).most_common(1)[0]
                final_pred = most_common
                chosen_idx = preds.index(most_common)
            else:
                final_pred = None
                chosen_idx = 0

        ok = is_correct(final_pred, gold)
        correct += ok
        records.append({
            "question": ex["question"],
            "gold": gold,
            "pred": final_pred,
            "all_preds": preds,
            "chosen_generation": gens[chosen_idx],
            "all_generations": gens if n_samples > 1 else None,
            "correct": ok,
        })

    acc = correct / len(eval_set)
    print(f"\n{label}: accuracy = {acc:.3%}  ({correct}/{len(eval_set)})  [{elapsed:.0f}s]")
    return {"accuracy": acc, "n_correct": correct, "n_total": len(eval_set),
            "elapsed_sec": elapsed, "records": records}

In [ ]:
# Sanity check
test_out = llm.generate(
    ["Q: 1+1=?\nA:"],
    SamplingParams(temperature=0, max_tokens=20)
)
print(test_out[0].outputs[0].text)

## Experiments


In [ ]:
greedy_params = SamplingParams(temperature=0.0, max_tokens=320, stop=["\nQ:"])
sc_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=320, n=20, stop=["\nQ:"])

prompts_io = [build_standard_prompt(ex["question"]) for ex in eval_set]
prompts_few_cot = [build_cot_prompt(ex["question"]) for ex in eval_set]
prompts_zero_cot = [build_zeroshot_cot_prompt(ex["question"]) for ex in eval_set]

t0 = time.time()
outputs_io = llm.generate(prompts_io, greedy_params)
outputs_few_cot = llm.generate(prompts_few_cot, greedy_params)
outputs_zero_cot = llm.generate(prompts_zero_cot, greedy_params)
outputs_few_cot_sc = llm.generate(prompts_few_cot, sc_params)
outputs_zero_cot_sc = llm.generate(prompts_zero_cot, sc_params)
print(f"All 5 conditions generated in {time.time() - t0:.0f}s")

## Results

In [ ]:
def make_records(eval_set, outputs, n_samples):
    correct, records = 0, []
    for ex, output in zip(eval_set, outputs):
        gens  = [o.text.strip() for o in output.outputs]
        preds = [extract_pred(g) for g in gens]
        gold  = extract_gold(ex["answer"])
        if n_samples == 1:
            final_pred, chosen_idx = preds[0], 0
        else:
            valid = [p for p in preds if p is not None]
            if valid:
                mc, _ = Counter(valid).most_common(1)[0]
                final_pred, chosen_idx = mc, preds.index(mc)
            else:
                final_pred, chosen_idx = None, 0
        ok = is_correct(final_pred, gold)
        correct += ok
        records.append({
            "question": ex["question"], "gold": gold, "pred": final_pred,
            "all_preds": preds, "chosen_generation": gens[chosen_idx],
            "all_generations": gens if n_samples > 1 else None, "correct": ok,
        })
    return {"accuracy": correct / len(eval_set),
            "n_correct": correct, "n_total": len(eval_set), "records": records}


result_io = make_records(eval_set, outputs_io, n_samples=1)
result_few_cot = make_records(eval_set, outputs_few_cot, n_samples=1)
result_zero_cot = make_records(eval_set, outputs_zero_cot, n_samples=1)
result_few_cot_sc = make_records(eval_set, outputs_few_cot_sc, n_samples=SC_SAMPLES)
result_zero_cot_sc = make_records(eval_set, outputs_zero_cot_sc, n_samples=SC_SAMPLES)


In [ ]:
all_results = {
    "standard_io": result_io,
    "zero_shot_cot": result_zero_cot,
    "few_shot_cot": result_few_cot,
    "zero_shot_cot_sc": result_zero_cot_sc,
    "few_shot_cot_sc": result_few_cot_sc,
}
for name, r in all_results.items():
    with open(RESULTS_DIR / f"{name}.json", "w") as f:
        json.dump(r, f, indent=2)

In [ ]:
io_acc = result_io["accuracy"]
print(f"\n{'Condition':<42} {'Accuracy':>10}   {'Δ vs IO':>10}")
print("-" * 67)
for label, r in [
    ("Standard IO (8-shot, greedy)", result_io),
    ("Zero-shot CoT (greedy)", result_zero_cot),
    ("Few-shot CoT (greedy)", result_few_cot),
    ("Zero-shot CoT + Self-Consistency (K=20)", result_zero_cot_sc),
    ("Few-shot CoT + Self-Consistency (K=20)", result_few_cot_sc),
]:
    change = r["accuracy"] - io_acc
    print(f"{label:<42} {r['accuracy']:>10.2%}   {change:>+10.2%}")

## Discussion


In [ ]:
def diff_records(a, b): return [i for i, (ra, rb) in enumerate(zip(a["records"], b["records"])) if ra["correct"] and not rb["correct"]]

cot_rescues_io  = diff_records(result_few_cot, result_io)
sc_rescues_cot  = diff_records(result_few_cot_sc, result_few_cot)

print(f"CoT fixed {len(cot_rescues_io)} problems that Standard IO missed.")
print(f"Self-Consistency fixed {len(sc_rescues_cot)} problems that greedy CoT missed.")

In [ ]:
io_rescues_cot  = diff_records(result_io,  result_few_cot)
cot_rescues_sc  = diff_records(result_few_cot, result_few_cot_sc)
print(f"(Reverse) Standard IO got {len(io_rescues_cot)} right that CoT missed.")
print(f"(Reverse) Greedy CoT got {len(cot_rescues_sc)} right that Self-Consistency missed.")

In [ ]:
def show_example(label, result_a, result_b, i, max_chars=600):
    print(f"=== {label} (problem #{i}) ===")
    print("Q:", result_a["records"][i]["question"])
    print("Gold:", result_a["records"][i]["gold"])
    print(f"Pred (winning method): {result_a['records'][i]['pred']}")
    print(f"Pred (losing method):  {result_b['records'][i]['pred']}")
    print("\nWinning generation:")
    print(result_a["records"][i]["chosen_generation"][:max_chars])
    print("\nLosing generation:")
    print(result_b["records"][i]["chosen_generation"][:max_chars])
    print()

if cot_rescues_io:
    show_example("CoT > IO", result_few_cot, result_io, cot_rescues_io[0])
if sc_rescues_cot:
    i = sc_rescues_cot[0]
    show_example("Self-Consistency > CoT", result_few_cot_sc, result_few_cot, i)
    valid_preds = [p for p in result_few_cot_sc["records"][i]["all_preds"] if p is not None]
    print("Vote distribution across samples:", Counter(valid_preds))


### K-sweep

In [ ]:
def vote_with_K_samples(outputs, eval_set, K):
    correct = 0
    for ex, output in zip(eval_set, outputs):
        gens_K  = [o.text.strip() for o in output.outputs[:K]]
        preds_K = [extract_pred(g) for g in gens_K]
        valid   = [p for p in preds_K if p is not None]
        if valid:
            mc, _ = Counter(valid).most_common(1)[0]
            ok = is_correct(mc, extract_gold(ex["answer"]))
        else:
            ok = False
        correct += ok
    return correct / len(eval_set)


K_values = [1, 3, 5, 7, 10, 15, 20]
ksweep_few  = {K: vote_with_K_samples(outputs_few_cot_sc,  eval_set, K) for K in K_values}
ksweep_zero = {K: vote_with_K_samples(outputs_zero_cot_sc, eval_set, K) for K in K_values}

with open(RESULTS_DIR / "ksweep.json", "w") as f:
    json.dump({"K_values": K_values,
               "few_shot_cot_sc":  ksweep_few,
               "zero_shot_cot_sc": ksweep_zero,
               "few_shot_cot_greedy":  result_few_cot["accuracy"],
               "zero_shot_cot_greedy": result_zero_cot["accuracy"]},
              f, indent=2)

print(f"{'K':>4}  {'Few-shot CoT+SC':>18}  {'Zero-shot CoT+SC':>18}")
print("-" * 50)
for K in K_values:
    print(f"{K:>4}  {ksweep_few[K]:>18.2%}  {ksweep_zero[K]:>18.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(K_values, [ksweep_few[K]  for K in K_values],
        "o-",  linewidth=2, markersize=7, label="Few-shot CoT + SC")
ax.plot(K_values, [ksweep_zero[K] for K in K_values],
        "s--", linewidth=2, markersize=7, label="Zero-shot CoT + SC")
ax.axhline(result_few_cot["accuracy"],  color="C0", linestyle=":",
           alpha=0.7, label="Few-shot CoT (greedy)")
ax.axhline(result_zero_cot["accuracy"], color="C1", linestyle=":",
           alpha=0.7, label="Zero-shot CoT (greedy)")
ax.axhline(result_io["accuracy"],       color="gray", linestyle=":",
           alpha=0.5, label="Standard IO")

ax.set_xlabel("Number of sampled reasoning trajectories (K)")
ax.set_ylabel("GSM8K accuracy (N=200)")
ax.set_title(f"Self-Consistency: accuracy vs sample budget\n({MODEL_NAME})")
ax.set_xticks(K_values)
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ksweep.png", dpi=150, bbox_inches="tight")
plt.show()